In [98]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [99]:
df=pd.read_csv("../../data/raw/major_leagues v1/sofascore_major_leagues_2526season.csv")
df_yearly=pd.read_csv("../../data/raw/major_leagues v1/sofascore_MLS_Argentina_26season.csv")

In [100]:
df.shape

(8056, 117)

In [101]:
df_yearly.shape

(1539, 117)

In [102]:
set(df.columns) == set(df_yearly.columns)

True

In [103]:
df.columns.tolist() == df_yearly.columns.tolist()

True

In [104]:
total_df = pd.concat([df, df_yearly], ignore_index=True)

In [105]:
total_df.shape

(9595, 117)

In [107]:
total_df = total_df.rename(columns={col: f"{col}".lower() for col in total_df.columns})

In [108]:
total_df=total_df.drop(columns=["outfielderblocks","totwappearances","goalsprevented"])

In [109]:
total_df["goals_minus_xg"] = total_df["goals"] - total_df["expectedgoals"]
total_df["assists_minus_xa"] = total_df["assists"] - total_df["expectedassists"]

C:\Users\vibha\AppData\Local\Temp\ipykernel_25008\2912888129.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["goals_minus_xg"] = total_df["goals"] - total_df["expectedgoals"]
C:\Users\vibha\AppData\Local\Temp\ipykernel_25008\2912888129.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["assists_minus_xa"] = total_df["assists"] - total_df["expectedassists"]


In [110]:
per90_cols = [
    # Finishing
    "goals",
    "expectedgoals",
    "goals_minus_xg",
    "totalshots",
    "shotsontarget",
    "shotsofftarget",
    "shotsfrominsidethebox",
    "shotsfromoutsidethebox",
    "hitwoodwork",
    "leftfootgoals",
    "rightfootgoals",
    "headedgoals",
    "goalsfrominsidethebox",
    "goalsfromoutsidethebox",
    "freekickgoal",
    "penaltiestaken",
    "penaltygoals",
    "bigchancesmissed",

    # Creativity
    "assists",
    "expectedassists",
    "assists_minus_xa",
    "goalsassistssum",
    "totalpasses",
    "accuratepasses",
    "inaccuratepasses",
    "totaloppositionhalfpasses",
    "accurateoppositionhalfpasses",
    "totalownhalfpasses",
    "accurateownhalfpasses",
    "accuratefinalthirdpasses",
    "keypasses",
    "totalattemptassist",
    "passtoassist",
    "bigchancescreated",
    "totallongballs",
    "accuratelongballs",
    "totalchippedpasses",
    "accuratechippedpasses",
    "totalcross",
    "accuratecrosses",

    # Possession
    "touches",
    "possessionlost",
    "dispossessed",
    "totalcontest",
    "successfuldribbles",
    "offsides",
    "wasfouled",
    "fouls",

    # Defensive
    "tackles",
    "tackleswon",
    "interceptions",
    "ballrecovery",
    "possessionwonattthird",
    "clearances",
    "blockedshots",
    "dribbledpast",
    "errorleadtoshot",
    "errorleadtogoal",

    # Duels
    "totalduelswon",
    "duellost",
    "groundduelswon",
    "aerialduelswon",
    "aeriallost",
    
    # Goalkeeping
    "saves",
    "savescaught",
    "savesparried",
    "savedshotsfrominsidethebox",
    "savedshotsfromoutsidethebox",
    "runsout",
    "successfulrunsout",
    "goalkicks",
    "punches",
    "highclaims",
    "crossesnotclaimed",  
    

    # Discipline
    "yellowcards",
    "yellowredcards",
    "directredcards",
    "owngoals"
]

rate_cols = [
    "goalconversionpercentage",
    "scoringfrequency",

    "accuratepassespercentage",
    "accuratelongballspercentage",
    "successfuldribblespercentage",

    "tackleswonpercentage",

    "totalduelswonpercentage",
    "groundduelswonpercentage",
    "aerialduelswonpercentage",

    "penaltyconversion",

    "rating",
    "totalrating",
    "countrating"
]

for col in per90_cols:
    total_df[f"{col}_per90"] = np.where(
        total_df["minutesplayed"] > 0,
        (total_df[col] / total_df["minutesplayed"]) * 90,
        np.nan
    )

total_df["goals_per_xg"] = np.where(
    total_df["expectedgoals"] > 0,
    total_df["goals"] / total_df["expectedgoals"],
    1
)

total_df["shots_on_target_pct"] = np.where(
    total_df["totalshots"] > 0,
    total_df["shotsontarget"] / total_df["totalshots"],
    0
)


total_df["inside_box_shot_pct"] = np.where(
    total_df["totalshots"] > 0,
    total_df["shotsfrominsidethebox"] / total_df["totalshots"],
    0
)

total_df["assist_conversion"] = np.where(
    total_df["keypasses"] > 0,
    total_df["assists"] / total_df["keypasses"],
    0
)


total_df["xa_per_keypass"] = np.where(
    total_df["keypasses"] > 0,
    total_df["expectedassists"] / total_df["keypasses"],
    0
)


total_df["final_third_pass_pct"] = np.where(
    total_df["accuratepasses"] > 0,
    total_df["accuratefinalthirdpasses"] / total_df["accuratepasses"],
    0
)

total_df["opp_half_pass_pct"] = np.where(
    total_df["accuratepasses"] > 0,
    total_df["accurateoppositionhalfpasses"] / total_df["accuratepasses"],
    0
)

total_df["dribbles_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["successfuldribbles"] / total_df["touches"],
    0
)

total_df["dispossessed_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["dispossessed"] / total_df["touches"],
    0
)

total_df["possession_lost_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["possessionlost"] / total_df["touches"],
    0
)

total_df["defensive_actions"] = (
    total_df["tackles"] +
    total_df["interceptions"] +
    total_df["ballrecovery"]
)

total_df["defensive_actions_per90"] = (
    total_df["tackles_per90"] +
    total_df["interceptions_per90"] +
    total_df["ballrecovery_per90"]
)

C:\Users\vibha\AppData\Local\Temp\ipykernel_25008\3909454853.py:118: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f"{col}_per90"] = np.where(
C:\Users\vibha\AppData\Local\Temp\ipykernel_25008\3909454853.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["goals_per_xg"] = np.where(
C:\Users\vibha\AppData\Local\Temp\ipykernel_25008\3909454853.py:130: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

In [111]:
total_df=total_df[total_df['minutesplayed'] > 900]

In [116]:
total_df['league'].value_counts()

league
England EFL Championship      436
Spain La Liga 2               387
Italy Serie B                 350
Spain La Liga                 347
England Premier League        341
Italy Serie A                 339
Turkiye Super Lig             289
France Ligue 2                288
Germany Bundesliga            285
Portugal Primeira Liga        282
France Ligue 1                277
Germany 2.Bundesliga          276
Saudi Arabia Pro League       270
Netherlands Eredivisie        268
Argentina Liga Profesional    239
USA MLS                       214
Name: count, dtype: int64

In [112]:
total_df.shape

(4888, 206)

In [113]:
metadata_features = [
    "player",
    "team",
    "player id",
    "team id",
    "league",
    "position",
    "minutesplayed",
    "matchesstarted",
    "appearances"
]

In [114]:
total_df=total_df.fillna(0.0)

In [115]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    null_values=total_df.isna().sum().sum()
    print(null_values)

0


In [117]:
total_rows = len(total_df)
unique_ids = total_df['player id'].nunique()
duplicate_count = total_rows - unique_ids
duplicate_count

16

In [118]:
features_to_scale = [
    col for col in total_df.columns 
    if total_df[col].dtype in [np.float64, np.int64] 
    and col not in metadata_features
]

for col in features_to_scale:
    grouped = total_df.groupby(["league", "position"])[col]
    group_mean = grouped.transform("mean")
    group_std = grouped.transform("std")
    
    total_df[f"{col}_zscore"] = np.where(
        group_std > 0,
        (total_df[col] - group_mean) / group_std,
        0.0
    )

C:\Users\vibha\AppData\Local\Temp\ipykernel_25008\1788466843.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f"{col}_zscore"] = np.where(


In [121]:
total_df.columns.to_list()

['accuratechippedpasses',
 'accuratecrosses',
 'accuratecrossespercentage',
 'accuratefinalthirdpasses',
 'accuratelongballs',
 'accuratelongballspercentage',
 'accurateoppositionhalfpasses',
 'accurateownhalfpasses',
 'accuratepasses',
 'accuratepassespercentage',
 'aerialduelswon',
 'aerialduelswonpercentage',
 'aeriallost',
 'appearances',
 'assists',
 'attemptpenaltymiss',
 'attemptpenaltypost',
 'attemptpenaltytarget',
 'ballrecovery',
 'bigchancescreated',
 'bigchancesmissed',
 'blockedshots',
 'cleansheet',
 'clearances',
 'countrating',
 'crossesnotclaimed',
 'directredcards',
 'dispossessed',
 'dribbledpast',
 'duellost',
 'errorleadtogoal',
 'errorleadtoshot',
 'expectedassists',
 'expectedgoals',
 'fouls',
 'freekickgoal',
 'goalconversionpercentage',
 'goalkicks',
 'goals',
 'goalsassistssum',
 'goalsconceded',
 'goalsconcededinsidethebox',
 'goalsconcededoutsidethebox',
 'goalsfrominsidethebox',
 'goalsfromoutsidethebox',
 'groundduelswon',
 'groundduelswonpercentage',
 'h

In [ ]:
total_df.drop(columns=per90_cols)